# Fine-Tuning GPT-2 for Joke Generation

This notebook fine-tunes GPT-2 on 1,622 short jokes so it can generate a full joke given the first 3 words.

## 1. Install Dependencies & Configuration

In [1]:
!pip install transformers torch accelerate -q
print("All dependencies installed")

All dependencies installed



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\kim83\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import csv
import random
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    GPT2LMHeadModel, GPT2Tokenizer, get_linear_schedule_with_warmup,
)

# CONFIG
DATA_PATH   = 'data'       
OUTPUT_DIR  = 'gpt2_jokes_model'
MODEL_NAME  = 'gpt2'              
MAX_LENGTH  = 128
BATCH_SIZE  = 8
EPOCHS      = 5
LR          = 5e-5
WARMUP      = 100
GRAD_ACCUM  = 2
VAL_SPLIT   = 0.1
BOS = '<|startoftext|>'
EOS = '<|endoftext|>'
PAD = '<|pad|>'

C:\Users\kim83\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load & preprocess the dataset

In [3]:
def load_jokes(path):
    jokes = []
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            j = row["Joke"].strip()
            if j:
                jokes.append(j)
    return jokes


jokes = load_jokes(DATA_PATH)
print(f"Total jokes: {len(jokes)}")
print("\nSample jokes:")
for j in random.sample(jokes, 5):
    print(f"  {j}")

Total jokes: 1622

Sample jokes:
  I was gonna make a joke on Reddit.. .. but I guess you've already Reddit somewhere.
  Why did the melon try so hard to get her father's approval? Because she cant-aloupe
  /r/pickle welcomes it's newest ally. It's always good to have clean jokes. I due urge the mods to add us to your sidebar, due to the fact that you are on ours.
  One fifth of people... ...are just too tense!
  I just heard because of the government shutdown government archeologists are working with a skeleton crew.


## 3. Set up GPT-2 tokenizer + model

In [4]:
# Wrap each joke: <|startoftext|> JOKE <|endoftext|>
formatted = [f"{BOS} {j} {EOS}" for j in jokes]

# Train/val split
random.seed(42)
random.shuffle(formatted)
split = int(len(formatted) * (1 - VAL_SPLIT))
train_jokes, val_jokes = formatted[:split], formatted[split:]
print(f"Train: {len(train_jokes)} | Val: {len(val_jokes)}")
print(f"\nExample formatted joke:\n  {formatted[0]}")

Train: 1459 | Val: 163

Example formatted joke:
  <|startoftext|> I just read this article about short term memory I don't remember what it was about <|endoftext|>


In [5]:
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.add_special_tokens({"bos_token": BOS, "eos_token": EOS, "pad_token": PAD})

model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
model.resize_token_embeddings(len(tokenizer))

n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {MODEL_NAME} ({n_params:,} parameters)")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 38064.57it/s]
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model: gpt2 (124,441,344 parameters)


## 4. Training

In [6]:
class JokeDataset(Dataset):
    def __init__(self, jokes, tokenizer, max_length):
        self.examples = []
        for joke in jokes:
            enc = tokenizer(
                joke,
                truncation=True,
                max_length=max_length,
                padding="max_length",
                return_tensors="pt",
            )
            ids = enc["input_ids"].squeeze()
            mask = enc["attention_mask"].squeeze()
            labels = ids.clone()
            labels[mask == 0] = -100  # ignore padding in loss
            self.examples.append(
                {"input_ids": ids, "attention_mask": mask, "labels": labels}
            )

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]


train_ds = JokeDataset(train_jokes, tokenizer, MAX_LENGTH)
val_ds = JokeDataset(val_jokes, tokenizer, MAX_LENGTH)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
print(f"Train batches: {len(train_dl)} | Val batches: {len(val_dl)}")

Train batches: 183 | Val batches: 21


In [7]:
total_steps = (len(train_dl) // GRAD_ACCUM) * EPOCHS
optimizer = AdamW(model.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(optimizer, WARMUP, total_steps)

best_val_loss = float("inf")
history = {"train": [], "val": []}
Path(OUTPUT_DIR).mkdir(exist_ok=True)

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    train_loss = 0
    optimizer.zero_grad()
    for step, batch in enumerate(train_dl):
        out = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"],
        )
        loss = out.loss / GRAD_ACCUM
        loss.backward()
        train_loss += out.loss.item()
        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    # Validate
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_dl:
            out = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )
            val_loss += out.loss.item()

    avg_train = train_loss / len(train_dl)
    avg_val = val_loss / len(val_dl)
    history["train"].append(avg_train)
    history["val"].append(avg_val)
    print(f"Epoch {epoch}/{EPOCHS}  |  Train: {avg_train:.4f}  |  Val: {avg_val:.4f}")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        model.save_pretrained(f"{OUTPUT_DIR}/best")
        tokenizer.save_pretrained(f"{OUTPUT_DIR}/best")
        print(f"Saved best model (val={best_val_loss:.4f})")

print(f"\n Best val loss: {best_val_loss:.4f}")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch 1/5  |  Train: 4.1077  |  Val: 3.4800


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.07s/it]


Saved best model (val=3.4800)
Epoch 2/5  |  Train: 3.2865  |  Val: 3.3995


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.03it/s]


Saved best model (val=3.3995)
Epoch 3/5  |  Train: 2.9387  |  Val: 3.4170
Epoch 4/5  |  Train: 2.7449  |  Val: 3.4436
Epoch 5/5  |  Train: 2.6244  |  Val: 3.4616

 Best val loss: 3.3995


## 5. Generate jokes

In [8]:
def generate_jokes(prompt, model_dir="gpt2_jokes_model/best", n=3):
    """Generate n jokes given a prompt string."""
    tok = GPT2Tokenizer.from_pretrained(model_dir)
    mdl = GPT2LMHeadModel.from_pretrained(model_dir)
    mdl.eval()

    input_ids = tok.encode(f"{BOS} {prompt}", return_tensors="pt")
    with torch.no_grad():
        outputs = mdl.generate(
            input_ids,
            max_length=100,
            temperature=0.9,
            top_k=50,
            top_p=0.95,
            do_sample=True,
            num_return_sequences=n,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
            repetition_penalty=1.3,
        )
    return [tok.decode(o, skip_special_tokens=True).strip() for o in outputs]

prompts = [
    # first 3 words from the data
    "What did the",
    "Don't you hate",
    "Why did the",
    # 3 random words
    "A dog is",        
    "My wife told",        
    "Purple monkey water",
]

for p in prompts:
    print(f'\n Prompt: "{p}"')
    print("─" * 50)
    for i, joke in enumerate(generate_jokes(p, n=2), 1):
        print(f"  [{i}] {joke}")


 Prompt: "What did the"
──────────────────────────────────────────────────


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4787.50it/s]


  [1] What did the rabbit say when it was at its best? Hey, I'm here for you.
  [2] What did the octopus say when it was hit with a baseball bat? "You've done everything!"

 Prompt: "Don't you hate"
──────────────────────────────────────────────────


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4218.36it/s]


  [1] Don't you hate them for their nose? They don.
  [2] Don't you hate when a girl gets into trouble? If she asks no, then they'll just walk away.

 Prompt: "Why did the"
──────────────────────────────────────────────────


...

  [1] A dog is born with a "P" and no legs.
  [2] A dog is dying.

 Prompt: "My wife told"
──────────────────────────────────────────────────


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4295.36it/s]


  [1] My wife told me the morning news... She said I was "out."
  [2] My wife told me she was on vacation... But I said "no, we'll take her back for a day".

 Prompt: "Purple monkey water"
──────────────────────────────────────────────────


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 3497.60it/s]


  [1] Purple monkey water.
  [2] Purple monkey watermelon juice.
